# text8 GPT

Training a decoder-only Transformer (grouped-query attention + RoPE + QK-norm) on text8, tokenized with the pretrained [LiquidAI/LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M) subword tokenizer.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import os
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from lightning import Trainer, seed_everything
from torchinfo import summary

from chimera.data import Text8DataModule
from chimera.models import GPT
from chimera.modules import LanguageModelModule
from chimera.optim import LinearWarmupCosineAnnealingLR, Muon, muon_param_groups

os.environ["DATA_DIR"] = "../../../data"

# Reproducibility: seed all RNGs (incl. dataloader workers). Pair with
# Trainer(deterministic=True) below for deterministic CUDA kernels too.
SEED = 42
seed_everything(SEED, workers=True)

SEQ_LEN = 2048

## Data

Load text8 and tokenize it with the pretrained **LiquidAI/LFM2.5-230M** subword tokenizer (vocab ≈ 64k) instead of the classic 27-symbol character vocabulary.

In [ ]:
dm = Text8DataModule(
    data_dir=os.environ["DATA_DIR"],
    batch_size=32,
    seq_len=SEQ_LEN,
    num_workers=0,
    pin_memory=False,
    tokenizer_backend="pretrained",  # LiquidAI/LFM2.5-230M
)
dm.prepare_data()
dm.setup("fit")

train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()

print(f"tokenizer={dm.pretrained_id}  vocab_size={dm.vocab_size}")
print(f"train sequences={len(dm.text8_train):,}  val sequences={len(dm.text8_val):,}")

In [ ]:
x, y = next(iter(train_loader))
print("sample:", repr(dm.decode(x[0][:64])))

# most frequent tokens in one batch, shown as their decoded text
counts = Counter(x.flatten().tolist())
top = counts.most_common(20)
labels = [repr(dm.decode([tid])) for tid, _ in top]
values = [c for _, c in top]

plt.figure(figsize=(12, 4))
plt.bar(range(len(top)), values)
plt.title("Top-20 Tokens (one training batch)")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(range(len(top)), labels, rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Model

A decoder-only Transformer with grouped-query attention, rotary position embeddings, and QK-norm. `block_size` must match the datamodule's `seq_len`.

In [ ]:
# round to multiple of 32
rounded_vocab_size = (dm.vocab_size + 31) // 32 * 32

model = GPT(
    vocab_size=rounded_vocab_size,
    block_size=SEQ_LEN,
    n_embd=48,
    n_head=2,
    n_kv_head=1,
    n_layer=6,
    tie_embedding=True,
)
summary(
    model,
    input_data=torch.zeros(1, SEQ_LEN, dtype=torch.long),
    col_names=["output_size", "num_params", "mult_adds"],
    depth=1,
)

## Training

Wrap the model in a `LightningModule` and train with a linear-warmup cosine-annealing schedule, minimizing next-token cross-entropy. One epoch over the 90M-character split is already substantial — raise `N_EPOCH` for a real run.

In [ ]:
N_EPOCH = 1
optimizer = Muon(
    muon_param_groups(
        model,
        muon_lr=0.02,
        adamw_lr=1e-3,
        adamw_weight_decay=0.1,
    )
)
scheduler = LinearWarmupCosineAnnealingLR(
    optimizer,
    warmup_steps=100,
    n_epochs=N_EPOCH,
    train_loader=train_loader,
)

model = torch.compile(model, mode="reduce-overhead")
# use_cce: fuse lm_head + cross-entropy (apple/ml-cross-entropy), no logits materialized
lm_module = LanguageModelModule(model, optimizer, scheduler, use_cce=True)

trainer = Trainer(
    max_epochs=N_EPOCH,
    precision="bf16-true",
    gradient_clip_val=1.0,
    deterministic=True,
)

trainer.fit(lm_module, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(lm_module, dataloaders=val_loader)

## Analysis

Plot the logged loss and bits-per-token curves, then sample text from the trained model.

In [ ]:
metrics = np.genfromtxt(
    f"{trainer.logger.log_dir}/metrics.csv", delimiter=",", names=True
)


def series(step_key, value_key):
    step = metrics[step_key]
    value = metrics[value_key]
    mask = ~np.isnan(value)
    return step[mask], value[mask]


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Training Metrics")

for ax, metric, title in zip(axes, ["loss", "bpt"], ["Loss (nats)", "Bits per Token"]):
    train_step, train_val = series("step", f"train{metric}")
    val_step, val_val = series("step", f"val{metric}")
    ax.plot(train_step, train_val, marker="o", label="train")
    ax.plot(val_step, val_val, marker="o", label="val")
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.show()

In [ ]:
from textwrap import fill

# Lightning moves the model to CPU after trainer.test(), so put it back on the GPU —
# otherwise generation runs on CPU and is ~30x slower.
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

prompt = "the meaning of life is "
idx = torch.tensor([dm.tokenizer.encode(prompt)], device=device)

# Eager by default (no compile warmup). Pass compile=True for ~2x faster decode
# when generating repeatedly — it costs a one-time ~10s compile on the first call.
generated = model.generate(idx, max_new_tokens=450, temperature=0.8)
print("\n".join(fill(dm.decode(generated[0].cpu()), width=80).splitlines()))